1. Dataset Preparation

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Создаем pipeline преобразований
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    # случайно отражает изображение → увеличивает разнообразие данных (data augmentation)

    transforms.ToTensor(),
    # переводит изображение в тензор (формат PyTorch)

    transforms.Normalize((0.5,), (0.5,))
    # нормализация → значения примерно в диапазоне [-1, 1], ускоряет обучение
])

# Загружаем тренировочные данные
train_data = datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

# Загружаем тестовые данные
test_data = datasets.MNIST(
    root='./data',
    train=False,
    transform=transform
)

# Создаем батчи (группы данных)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
# batch_size=64 → обрабатываем по 64 изображения за раз
# shuffle=True → перемешиваем данные (важно для обучения)

test_loader = DataLoader(test_data, batch_size=64)

2. Building the CNN

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Первый сверточный слой
        self.conv1 = nn.Conv2d(
            in_channels=1,   # вход: grayscale → 1 канал
            out_channels=16, # 16 фильтров → 16 feature maps
            kernel_size=3,   # размер фильтра 3x3
            padding=1        # сохраняем размер изображения
        )

        # Второй сверточный слой
        self.conv2 = nn.Conv2d(
            16,  # вход = 16 feature maps
            32,  # выход = 32 feature maps
            3,
            padding=1
        )

        # Пулинг слой
        self.pool = nn.MaxPool2d(2,2)
        # уменьшает размер в 2 раза

        # Полносвязный слой
        self.fc = nn.Linear(32*7*7, 10)
        # 32 feature maps по 7x7 → 32*7*7 входов
        # 10 выходов (10 классов цифр)

    def forward(self, x):

        # --- Первый блок ---
        x = self.conv1(x)
        # применяем фильтры → ищем линии

        x = F.relu(x)
        # убираем отрицательные значения

        x = self.pool(x)
        # уменьшаем размер

        # --- Второй блок ---
        x = self.conv2(x)
        # ищем более сложные признаки

        x = F.relu(x)

        x = self.pool(x)

        # --- Flatten ---
        x = x.view(x.size(0), -1)
        # превращаем в вектор

        # --- Linear ---
        x = self.fc(x)
        # финальное предсказание

        return x

3. Training the CNN

In [ ]:
import torch.optim as optim

model = CNN()

# функция потерь
criterion = nn.CrossEntropyLoss()
# используется для классификации

# оптимизатор
optimizer = optim.Adam(model.parameters(), lr=0.001)
# обновляет веса модели

for epoch in range(5):  # 5 эпох
    total_loss = 0

    for images, labels in train_loader:

        optimizer.zero_grad()
        # обнуляем старые градиенты

        outputs = model(images)
        # forward pass → получаем предсказания

        loss = criterion(outputs, labels)
        # считаем ошибку

        loss.backward()
        # backpropagation → считаем градиенты

        optimizer.step()
        # обновляем веса

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 339.8902
Epoch 2, Loss: 149.9530
Epoch 3, Loss: 113.8821
Epoch 4, Loss: 96.3941
Epoch 5, Loss: 85.1129


4. Visualizing Feature Maps

In [ ]:
import matplotlib.pyplot as plt

def visualize_feature_maps(x, model):

    with torch.no_grad():  # отключаем обучение

        x = model.conv1(x)
        fmap1 = x  # после conv1

        x = F.relu(x)
        fmap2 = x  # после ReLU

        x = model.pool(x)
        fmap3 = x  # после pooling

        x = model.conv2(x)
        fmap4 = x

        x = F.relu(x)
        fmap5 = x

        x = model.pool(x)
        fmap6 = x

    maps = [fmap1, fmap2, fmap3, fmap4, fmap5, fmap6]

    for i, fmap in enumerate(maps):
        plt.figure(figsize=(6,6))

        for j in range(min(6, fmap.shape[1])):
            plt.subplot(2,3,j+1)
            plt.imshow(fmap[0][j].cpu(), cmap='gray')
            plt.axis('off')

        plt.title(f"Layer {i+1}")
        plt.show()

5. Model Evaluation

In [ ]:
correct = 0
total = 0

with torch.no_grad():  # не обучаем модель
    for images, labels in test_loader:

        outputs = model(images)
        # получаем предсказания

        _, predicted = torch.max(outputs, 1)
        # выбираем класс с максимальным значением

        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        # считаем правильные ответы

accuracy = correct / total

print("Accuracy:", accuracy)

Accuracy: 0.9689


## Architecture & Design

### Why do we use small kernels (3×3) instead of large ones?

Small kernels (like 3×3) reduce the number of parameters and computational cost. Stacking multiple small kernels increases the depth of the network and allows it to learn more complex features.

**Simple idea:** Two 3×3 layers ≈ one large filter, but more efficient.

### What is the difference between "Valid" and "Same" padding?

* **Valid padding**: no padding → output size decreases
* **Same padding**: padding is added → output size stays the same

---

## Activation Functions

### Why is ReLU preferred over Sigmoid?

ReLU avoids the vanishing gradient problem and allows faster training. Sigmoid compresses values into a small range, leading to very small gradients.

**Simple idea:** Sigmoid slows learning, ReLU keeps strong signals.

### What happens when ReLU outputs 0?

If ReLU outputs 0, the gradient is also 0, so the neuron stops learning. This is known as the "dying ReLU" problem.

---

## Pooling

### What is the purpose of Max Pooling?

Max pooling reduces spatial dimensions while preserving the most important features.

### Does pooling have trainable parameters?

No, pooling layers do not have trainable parameters.

### How does pooling contribute to translational invariance?

Pooling allows the model to recognize features even if their position slightly changes in the image.

---

## Training & Regularization

### What if training accuracy is high but test accuracy is low?

This is overfitting. The model memorizes training data but does not generalize well to new data.

### How to fix overfitting?

* Use more data
* Apply dropout
* Use regularization

### How does Dropout help?

Dropout randomly disables neurons during training, forcing the model to learn more robust and general features.

---

## Theoretical Intuition

### What do early vs deep layers learn?

* Early layers learn simple features (edges, lines)
* Deeper layers learn complex features (shapes, objects)

### What is Receptive Field?

Receptive field is the region of the input image that influences a neuron. It increases as we go deeper into the network.

**Simple idea:** Early layers see small areas, deeper layers see larger parts of the image.

---

## Final Summary

CNN learns hierarchical features: simple patterns in early layers and complex representations in deeper layers.


## Архитектура и дизайн

### Почему используются маленькие фильтры (3×3)?

Маленькие фильтры уменьшают количество параметров и вычислений. Несколько слоев 3×3 позволяют строить более глубокую сеть и изучать сложные признаки.

**Просто:** Два слоя 3×3 ≈ один большой фильтр, но эффективнее.

### Разница между "Valid" и "Same" padding

* **Valid**: без padding → размер уменьшается
* **Same**: добавляется padding → размер сохраняется

---

## Функции активации

### Почему ReLU лучше, чем Sigmoid?

ReLU не вызывает исчезновение градиента и обучается быстрее. Sigmoid сжимает значения, из-за чего градиенты становятся маленькими.

**Просто:** Sigmoid тормозит обучение, ReLU пропускает сигнал.

### Что происходит, если ReLU = 0?

Если ReLU дает 0, градиент тоже равен 0, и нейрон перестает обучаться ("мертвый ReLU").

---

## Pooling

### Зачем нужен Max Pooling?

Max pooling уменьшает размер изображения, сохраняя важные признаки.

### Есть ли у pooling параметры?

Нет, pooling не имеет обучаемых параметров.

### Как pooling дает устойчивость к смещению?

Pooling позволяет распознавать объекты, даже если они немного смещены.

---

## Обучение и регуляризация

### Высокая точность на train, низкая на test

Это переобучение — модель запоминает данные, но плохо работает на новых.

### Как исправить?

* больше данных
* dropout
* регуляризация

### Что делает Dropout?

Dropout случайно отключает нейроны, заставляя сеть учиться более устойчивым признакам.

---

## Теория

### Что изучают ранние и глубокие слои?

* Ранние слои — простые признаки (линии)
* Глубокие слои — сложные объекты

### Что такое Receptive Field?

Это область изображения, которую "видит" нейрон. С глубиной сети она увеличивается.

**Просто:** Первый слой видит маленький участок, глубокий — почти всю картинку.

---

## Итог

CNN обучается по иерархии: сначала простые признаки, затем сложные.
